# Face Indexer — Google Colab
Chạy notebook này trên Colab GPU để đánh index khuôn mặt nhanh hơn.
Kết quả tự động push thẳng vào app qua ngrok.

In [ ]:
# ── Cài thư viện ──────────────────────────────────────────────────────────────
!pip install -q deepface tf-keras ultralytics google-api-python-client google-auth requests

In [ ]:
# ── Cấu hình ──────────────────────────────────────────────────────────────────
FOLDER_IDS = [
    "PASTE_FOLDER_ID_HERE",
]

CREDENTIALS_PATH = "/content/credentials.json"
YOLO_MODEL_PATH  = "/content/yolov11_face.pt"
OUTPUT_PATH      = "/content/drive/MyDrive/face_index.json"

# URL app của bạn qua ngrok (xem hướng dẫn cell bên dưới)
APP_URL  = "https://xxxx-xx-xx-xx-xx.ngrok-free.app"  # thay bằng URL ngrok thực
API_KEY  = "my-secret-key-123"  # phải khớp với PUSH_API_KEY trong docker-compose.yml

CONF_THRESHOLD = 0.25
FACE_PAD       = 0.15
EMBED_MODEL    = "Facenet512"
PUSH_BATCH     = 20   # push lên app sau mỗi N ảnh

## Hướng dẫn cấu hình ngrok
1. Tạo tài khoản miễn phí tại https://ngrok.com
2. Lấy authtoken tại https://dashboard.ngrok.com/get-started/your-authtoken
3. Chạy trên máy tính (nơi Docker đang chạy):
```bash
brew install ngrok   # hoặc: https://ngrok.com/download
ngrok config add-authtoken YOUR_TOKEN
ngrok http 8000
```
4. Copy URL dạng `https://xxxx.ngrok-free.app` điền vào `APP_URL` ở trên

In [ ]:
# ── Mount Google Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Upload credentials.json và yolov11_face.pt ────────────────────────────────
from google.colab import files
print("Upload credentials.json:")
files.upload()
print("\nUpload yolov11_face.pt:")
files.upload()

In [ ]:
# ── List ảnh đệ quy tất cả thư mục con ──────────────────────────────────────
IMAGE_MIME  = {'image/jpeg', 'image/png', 'image/webp', 'image/heic', 'image/bmp'}
FOLDER_MIME = 'application/vnd.google-apps.folder'

def list_all_recursive(root_id, _depth=0):
    """Trả về list tất cả file ảnh trong root_id và các thư mục con (đệ quy)."""
    images, subfolders, token = [], [], None

    while True:
        r = drive_svc.files().list(
            q=f"'{root_id}' in parents and trashed=false",
            fields="nextPageToken,files(id,name,mimeType)",
            pageSize=1000, pageToken=token,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute()
        for f in r.get('files', []):
            if f['mimeType'] == FOLDER_MIME:
                subfolders.append(f)
            elif f['mimeType'] in IMAGE_MIME:
                f['folder_id'] = root_id
                images.append(f)
        token = r.get('nextPageToken')
        if not token:
            break

    indent = '  ' * _depth
    print(f"{indent}📁 {root_id}: {len(images)} ảnh, {len(subfolders)} thư mục con")

    for sub in subfolders:
        images += list_all_recursive(sub['id'], _depth + 1)

    return images

all_files = []
for fid in FOLDER_IDS:
    print(f"\nQuét folder gốc: {fid}")
    imgs = list_all_recursive(fid)
    all_files.extend(imgs)

# Loại bỏ trùng lặp theo file_id
seen = set()
unique_files = []
for f in all_files:
    if f['id'] not in seen:
        seen.add(f['id'])
        unique_files.append(f)
all_files = unique_files

print(f"\n📷 Tổng cộng: {len(all_files)} ảnh từ tất cả thư mục")

In [ ]:
# ── Test kết nối tới app ──────────────────────────────────────────────────────
import requests
try:
    r = requests.get(f"{APP_URL}/api/health", timeout=10)
    print(f"✅ App online: {r.json()}")
except Exception as e:
    print(f"❌ Không kết nối được app: {e}")
    print("Kiểm tra lại APP_URL và ngrok")

In [ ]:
# ── List ảnh từ tất cả folders ───────────────────────────────────────────────
IMAGE_MIME = {'image/jpeg', 'image/png', 'image/webp', 'image/heic'}

def list_images(folder_id):
    files, token = [], None
    while True:
        r = drive_svc.files().list(
            q=f"'{folder_id}' in parents and trashed=false",
            fields="nextPageToken,files(id,name,mimeType)",
            pageSize=1000, pageToken=token
        ).execute()
        files += [f for f in r.get('files', []) if f['mimeType'] in IMAGE_MIME]
        token = r.get('nextPageToken')
        if not token: break
    return files

all_files = []
for fid in FOLDER_IDS:
    imgs = list_images(fid)
    for img in imgs:
        img['folder_id'] = fid
    all_files.extend(imgs)
    print(f"Folder {fid}: {len(imgs)} ảnh")
print(f"\n📷 Tổng: {len(all_files)} ảnh")

In [ ]:
# ── Chạy index + lưu file JSON ───────────────────────────────────────────────
import json, os
from tqdm.notebook import tqdm

DRIVE_VIEW = "https://drive.google.com/file/d/{}/view"

# Resume nếu đã chạy dở
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH) as f:
        results = json.load(f)
    done_ids = {r['file_id'] for r in results}
    print(f"Resume: đã có {len(results)} ảnh, bỏ qua {len(done_ids)} ảnh đã xử lý")
else:
    results, done_ids = [], set()

to_process = [f for f in all_files if f['id'] not in done_ids]
print(f"Cần xử lý: {len(to_process)} ảnh")

errors = []
for f in tqdm(to_process, desc="Indexing"):
    try:
        img = download_image(f['id'])
        if img is None:
            errors.append({'file': f['name'], 'error': 'cannot decode'})
            continue
        emb = extract_embedding(img)
        results.append({
            'file_id':    f['id'],
            'file_name':  f['name'],
            'folder_id':  f['folder_id'],
            'drive_link': DRIVE_VIEW.format(f['id']),
            'embedding':  emb,
        })
    except Exception as e:
        errors.append({'file': f['name'], 'error': str(e)})

    # Auto-save mỗi 50 ảnh
    if len(results) % 50 == 0:
        with open(OUTPUT_PATH, 'w') as out:
            json.dump(results, out)

# Lưu lần cuối
with open(OUTPUT_PATH, 'w') as out:
    json.dump(results, out)

indexed = sum(1 for r in results if r['embedding'])
print(f"\n✅ Xong! {indexed}/{len(results)} ảnh có khuôn mặt")
print(f"❌ Lỗi: {len(errors)}")
print(f"📁 Đã lưu: {OUTPUT_PATH}")

In [ ]:
# ── Hàm xử lý ────────────────────────────────────────────────────────────────
DRIVE_VIEW = "https://drive.google.com/file/d/{}/view"

def download_image(file_id):
    req = drive_svc.files().get_media(fileId=file_id)
    buf = io.BytesIO()
    dl  = MediaIoBaseDownload(buf, req)
    done = False
    while not done:
        _, done = dl.next_chunk()
    buf.seek(0)
    arr = np.frombuffer(buf.read(), np.uint8)
    return cv2.imdecode(arr, cv2.IMREAD_COLOR)


def extract_embedding(img_bgr):
    h, w = img_bgr.shape[:2]
    results = yolo(img_bgr, conf=CONF_THRESHOLD, verbose=False)
    boxes   = results[0].boxes
    if boxes is None or len(boxes) == 0:
        return None
    crops = []
    for box in boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = box[:4]
        pw = (x2-x1)*FACE_PAD; ph = (y2-y1)*FACE_PAD
        x1,y1,x2,y2 = max(0,int(x1-pw)),max(0,int(y1-ph)),min(w,int(x2+pw)),min(h,int(y2+ph))
        crops.append(img_bgr[y1:y2, x1:x2])
    crop = max(crops, key=lambda c: c.shape[0]*c.shape[1])
    crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    res = DeepFace.represent(crop_rgb, model_name=EMBED_MODEL,
                             detector_backend='skip', enforce_detection=False, align=False)
    return res[0]['embedding'] if res else None


def push_to_app(batch):
    """Push batch embeddings lên app."""
    r = requests.post(
        f"{APP_URL}/api/sync/push",
        json={"records": batch, "api_key": API_KEY},
        timeout=60
    )
    r.raise_for_status()
    return r.json()

In [ ]:
# ── Chạy index + tự động push ────────────────────────────────────────────────
import json, os
from tqdm.notebook import tqdm

# Load kết quả cũ nếu có (resume)
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH) as f:
        all_results = json.load(f)
    done_ids = {r['file_id'] for r in all_results}
    print(f"Resume: đã có {len(all_results)} ảnh")
else:
    all_results, done_ids = [], set()

to_process = [f for f in all_files if f['id'] not in done_ids]
print(f"Cần xử lý: {len(to_process)} ảnh")

batch, errors = [], []
total_pushed = 0

for f in tqdm(to_process, desc="Indexing"):
    try:
        img = download_image(f['id'])
        if img is None:
            errors.append({'file': f['name'], 'error': 'cannot decode'}); continue
        emb = extract_embedding(img)
        rec = {
            'file_id':    f['id'],
            'file_name':  f['name'],
            'folder_id':  f['folder_id'],
            'drive_link': DRIVE_VIEW.format(f['id']),
            'embedding':  emb,
        }
        batch.append(rec)
        all_results.append(rec)
    except Exception as e:
        errors.append({'file': f['name'], 'error': str(e)})

    # Push batch lên app
    if len(batch) >= PUSH_BATCH:
        try:
            push_to_app(batch)
            total_pushed += len(batch)
            print(f"  ✅ Pushed {total_pushed} ảnh lên app")
            batch = []
        except Exception as e:
            print(f"  ⚠️ Push lỗi: {e} — sẽ thử lại sau")

    # Auto-save backup
    if len(all_results) % 50 == 0:
        with open(OUTPUT_PATH, 'w') as out:
            json.dump(all_results, out)

# Push phần còn lại
if batch:
    try:
        push_to_app(batch)
        total_pushed += len(batch)
    except Exception as e:
        print(f"⚠️ Push cuối lỗi: {e}")

with open(OUTPUT_PATH, 'w') as out:
    json.dump(all_results, out)

indexed = sum(1 for r in all_results if r['embedding'])
print(f"\n✅ Xong! Index: {indexed}/{len(all_results)} | Push: {total_pushed} | Lỗi: {len(errors)}")